# Experiment 5 - LLM zero-/few-shot baselines

Prompt small open-weight instruction models to classify depression severity (1 minimal ... 4 severe)
on the shared Bengali test split, without any fine-tuning. The model answers with one digit; the
prediction is the most probable digit among the next-token probabilities (`src/llm.py`).

| Condition | Input | Shots per class |
|---|---|---|
| `bn` | native Bengali test posts | 0 or 2 |
| `bn2en` | the same posts translated with `facebook/nllb-200-distilled-600M` | 0 or 2 |
| `en` | English test split (sanity check only, different dataset) | 0 |

Zero-shot is deterministic (seed 42, run once). Two-shot runs use seeds 42, 43, 44 to vary only the
selected demonstrations (drawn from a 100-post training-only pool).

**How the runs were originally made.** Most runs were done locally, one command per run:

```
py -3 run_llm.py --condition bn --model Qwen/Qwen2.5-3B-Instruct --shots 0 --seed 42
py -3 run_llm.py --condition bn2en --model Qwen/Qwen2.5-3B-Instruct --shots 2 --seed 43
py -3 scripts\summarize_experiment5.py
```

This notebook runs exactly the same commands on a Colab GPU (4-bit via bitsandbytes). Gemma and
Llama are gated on Hugging Face: accept their licences first and set `HF_TOKEN` in Colab Secrets.
Finished runs are skipped, so the notebook can be re-run after a runtime restart.

In [ ]:
# Runtime check: this experiment needs a GPU.
import torch, transformers

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device       :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU found. In Colab go to Runtime > Change runtime type > "
        "Hardware accelerator = GPU, then re-run this cell."
    )

In [ ]:
# Install pinned deps. torch is deliberately NOT installed so Colab's CUDA build stays intact.
# bitsandbytes = 4-bit loading; sentencepiece = NLLB tokenizer (translation step).
!pip install -q "transformers==5.18.0" accelerate bitsandbytes sentencepiece openpyxl pandas scikit-learn

In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iammatthi/NLP.git"], check=True)
os.chdir("NLP")
print("working dir:", os.getcwd())

data_file = pathlib.Path("data/raw/depression-severity-bengali.xlsx")
assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")

In [ ]:
# Hugging Face login (needed for the gated Gemma / Llama weights).
# Set HF_TOKEN in Colab Secrets (key icon in the left sidebar); it is never printed.
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN: read from Colab Secrets")
except Exception:
    print("HF_TOKEN: no Colab Secret found, falling back to a login prompt")

from huggingface_hub import login, whoami
if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()
print("authenticated as:", whoami()["name"])

In [ ]:
# Stratified 80/10/10 splits, fixed seed (regenerated every session, byte-reproducible).
# bn is needed for all conditions; en only for the English sanity check.
!python scripts/prepare_splits.py --lang bn --seed 42
!python scripts/prepare_splits.py --lang en --seed 42

In [ ]:
# NLLB translations of the Bengali test split and the 100-post few-shot pool (bn2en).
# Skipped when the files are already in the repo.
import pathlib
needed = [pathlib.Path("data/translated/bn_test_en.csv"), pathlib.Path("data/translated/bn_pool_en.csv")]
if all(p.exists() for p in needed):
    print("translations present:", [str(p) for p in needed])
else:
    !python scripts/translate_bn_to_en.py

In [ ]:
# Run configuration. Same CLI as run_llm.py; finished runs (eval_test.json present) are skipped.
import subprocess, pathlib

MODELS = [
    "google/gemma-3-4b-it",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
]
FEWSHOT_SEEDS = [42, 43, 44]
RESULTS = pathlib.Path("results/experiment5")

def tag(model):
    return model.rsplit("/", 1)[-1].lower()

def run(model, condition, shots, seed, limit=None):
    name = f"{tag(model)}_{condition}_{shots}shot_seed{seed}" + ("_smoke" if limit else "")
    if (RESULTS / name / "eval_test.json").exists():
        print("skip (done):", name)
        return
    cmd = ["python", "run_llm.py", "--condition", condition, "--model", model,
           "--shots", str(shots), "--seed", str(seed)]
    if limit:
        cmd += ["--limit", str(limit)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

## 1. Smoke test and prompt-length check

**Run the prompt-length check before any `bn` few-shot run.** `HFScorer` truncates prompts at
`max_input_tokens=2048` from the right, which would cut off the query post and the answer prompt.
Bengali needs many more tokens per character than English, so eight Bengali examples plus the query
may not fit. If `truncated` is above 0 %, raise `max_input_tokens` in `src/llm.py` and rerun the
affected `bn` two-shot runs (delete their folders first, the cache is resumable).

In [ ]:
# 20-post smoke test (writes *_smoke folders, which are gitignored).
run(MODELS[0], "bn2en", 0, 42, limit=20)

In [ ]:
# Prompt length vs. the 2048-token limit, per model (tokenizers differ a lot on Bengali).
# Zero-shot first (longest single post), then two-shot (8 examples + query).
for model in MODELS:
    !python scripts/check_prompt_lengths.py --model {model} --shots 0
    !python scripts/check_prompt_lengths.py --model {model} --shots 2

## 2. Zero-shot runs (deterministic, seed 42)

In [ ]:
for model in MODELS:
    for condition in ("bn", "bn2en"):
        run(model, condition, 0, 42)

## 3. Two-shot runs (seeds 42, 43, 44)

Do the `bn2en` runs first. Run the `bn` runs only once the prompt-length check above is clean.

In [ ]:
for model in MODELS:
    for seed in FEWSHOT_SEEDS:
        run(model, "bn2en", 2, seed)

In [ ]:
for model in MODELS:
    for seed in FEWSHOT_SEEDS:
        run(model, "bn", 2, seed)

In [ ]:
# English sanity check (zero-shot, different dataset - not a language-only comparison).
run("Qwen/Qwen2.5-3B-Instruct", "en", 0, 42)

## 4. Paired translation drift (bn vs bn2en) and summary

In [ ]:
# McNemar + transition matrices for each model, same posts native vs translated.
(RESULTS / "translation_drift").mkdir(parents=True, exist_ok=True)

def drift(model, shots, seed):
    t = tag(model)
    bn = RESULTS / f"{t}_bn_{shots}shot_seed{seed}" / "predictions.csv"
    en = RESULTS / f"{t}_bn2en_{shots}shot_seed{seed}" / "predictions.csv"
    if not (bn.exists() and en.exists()):
        print("missing runs for", t, shots, seed)
        return
    out = RESULTS / "translation_drift" / f"{t}_{shots}shot_seed{seed}_translation_drift.json"
    subprocess.run(["python", "scripts/analyze_translation_drift.py",
                    "--bn", str(bn), "--bn2en", str(en), "--out", str(out)],
                   check=True, stdout=subprocess.DEVNULL)
    print("wrote", out)

for model in MODELS:
    drift(model, 0, 42)
    for seed in FEWSHOT_SEEDS:
        drift(model, 2, seed)

In [ ]:
# Regenerate results/experiment5/README.md from all completed runs.
!python scripts/summarize_experiment5.py
print(open("results/experiment5/README.md", encoding="utf-8").read()[:3000])

In [ ]:
# Colab's VM is wiped when the runtime recycles: download the results (raw.jsonl caches excluded).
import shutil
from google.colab import files

shutil.rmtree("exp5_results_tmp", ignore_errors=True)
shutil.copytree("results/experiment5", "exp5_results_tmp",
                ignore=shutil.ignore_patterns("raw.jsonl", "*_smoke"))
shutil.make_archive("experiment5_results", "zip", "exp5_results_tmp")
files.download("experiment5_results.zip")